In [ ]:
import json
import re
from pathlib import Path
from datetime import date

# Этап 1: txt-файлы с дашбордов -> JSON

Минимальный вариант: только чтение и парсинг, без Excel и без справочника.

1. Ищет все `.txt`-файлы рекурсивно внутри папки `txt/` (в любой вложенной подпапке — не привязано к дате).
2. Дэш и Экран определяются из **имени файла** (`<Дэш>_<Экран>.txt`).
3. Каждый файл разбирается на виджеты (показатели) тем же парсером, что и раньше.
4. Результат сохраняется в один `combined_widgets.json` — его на следующем этапе будем отправлять в LLM.

**Зависимости не нужны вообще** — только стандартная библиотека Python.

In [ ]:
MONTHS_RU = {
    "янв": 1, "января": 1, "январь": 1,
    "фев": 2, "февраля": 2, "февраль": 2,
    "мар": 3, "марта": 3, "март": 3,
    "апр": 4, "апреля": 4, "апрель": 4,
    "май": 5, "мая": 5,
    "июн": 6, "июня": 6, "июнь": 6,
    "июл": 7, "июля": 7, "июль": 7,
    "авг": 8, "августа": 8, "август": 8,
    "сен": 9, "сентября": 9, "сентябрь": 9,
    "окт": 10, "октября": 10, "октябрь": 10,
    "ноя": 11, "ноября": 11, "ноябрь": 11,
    "дек": 12, "декабря": 12, "декабрь": 12,
}

# месяц с явным годом: "авг2025", "янв.24", "январь 2024", "01.2024", "2024-01"
MONTH_WITH_YEAR_RE = re.compile(
    r"^(?:"
    r"(?P<name>[а-яё]+)\.?\s*['`]?(?P<y1>\d{2,4})"
    r"|(?P<mm>\d{1,2})[./-](?P<y2>\d{2,4})"
    r"|(?P<y3>\d{4})[./-](?P<mm2>\d{1,2})"
    r")$",
    re.IGNORECASE,
)
BARE_MONTH_RE = re.compile(r"^[а-яё]+$", re.IGNORECASE)

# квартал: "3Q2025", "4Q", "1кв2026", "2 кв. 25"
QUARTER_RE = re.compile(
    r"^(?P<num>[1-4])\s*(?:q|кв)\.?\s*(?P<year>\d{2,4})?$",
    re.IGNORECASE,
)

NUMBER_RE = re.compile(r"^-?\d[\d\s.,]*%?$")
HAS_DIGIT_RE = re.compile(r"\d")  # используется для проверки недельного среза - хватает одной цифры

FORECAST_KEYWORDS = ("прогноз",)
WEEKLY_KEYWORDS = ("нед",)

# единицы измерения (руб может быть сокращён до одной буквы "Р")
UNIT_PATTERNS = [
    re.compile(r"^(млн|тыс|млрд|трлн)\.?\s*(руб|р)\.?$", re.IGNORECASE),
    re.compile(r"^(руб|р)\.?$", re.IGNORECASE),
    re.compile(r"^%$"),
    re.compile(r"^(млн|тыс|млрд|трлн)?\.?\s*шт\.?$", re.IGNORECASE),
    re.compile(r"^(млн|тыс|млрд|трлн)?\.?\s*ед\.?$", re.IGNORECASE),
    # добавьте сюда свои варианты единиц измерения, если парсер их не находит
]

# служебные слова-подписи (легенда/KPI-плашки) - сигнализируют, что заголовок только что закончился
LABEL_STEMS = ("прогноз", "план", "факт", "выполнение", "вып", "дельта")


def is_unit_line(token: str) -> bool:
    t = token.strip()
    return any(p.match(t) for p in UNIT_PATTERNS)


def _normalize_label(token: str) -> str:
    t = token.strip().lower()
    t = re.sub(r"[.\-):]+$", "", t)
    t = re.sub(r"^[.\-(:]+", "", t)
    return t


def is_label_or_unit_line(token: str) -> bool:
    if is_unit_line(token):
        return True
    t = _normalize_label(token)
    return any(t == stem or t.startswith(stem) for stem in LABEL_STEMS)


def _month_name_to_num(name: str):
    name = name.lower()
    for key, num in MONTHS_RU.items():
        if name.startswith(key):
            return num
    return None


def parse_period_token(token: str, current_year):
    '''
    Пытается распознать токен как месяц или квартал (с явным годом или без - тогда
    используется "протянутый" current_year). Возвращает ((тип, год, номер), новый_год)
    либо (None, current_year). тип = "M" (месяц) или "Q" (квартал).
    '''
    t = token.strip().lower()

    m = MONTH_WITH_YEAR_RE.match(t)
    if m:
        if m.group("name"):
            month = _month_name_to_num(m.group("name"))
            year = m.group("y1")
        elif m.group("mm"):
            month = int(m.group("mm"))
            year = m.group("y2")
        else:
            month = int(m.group("mm2"))
            year = m.group("y3")
        if month is not None and 1 <= month <= 12:
            year = int(year)
            if year < 100:
                year += 2000
            return ("M", year, month), year

    q = QUARTER_RE.match(t)
    if q:
        num = int(q.group("num"))
        year = q.group("year")
        if year is not None:
            year = int(year)
            if year < 100:
                year += 2000
            return ("Q", year, num), year
        elif current_year is not None:
            return ("Q", current_year, num), current_year

    if BARE_MONTH_RE.match(t) and current_year is not None:
        month = _month_name_to_num(t)
        if month is not None:
            return ("M", current_year, month), current_year

    return None, current_year


def split_line(line: str):
    if "\t" in line:
        parts = line.split("\t")
    else:
        parts = re.split(r"\s{2,}", line)
    return [p.strip() for p in parts if p.strip() != ""]


def to_number(s: str):
    s = s.replace("\xa0", "").replace(" ", "").replace("%", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def flatten_tokens(text: str):
    tokens = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        tokens.extend(split_line(line))
    return tokens


def format_period(p):
    ptype, year, num = p
    if ptype == "M":
        return f"{num:02d}.{year}"
    return f"{num}кв.{year}"


def period_sort_key(p):
    return (p[1], p[2])

## Парсер

Правила:

- **Заголовок первого виджета в файле:**
  - для Дэшей, чьё имя содержит **"CB"** или **"core"** — заголовок это строка перед **первой строкой с цифрой**;
  - для Дэшей с **"УБ"** и всех остальных — строка перед первым служебным словом (единица измерения / Прогноз / План / Факт / Вып / Дельта).
- **Заголовки последующих виджетов** — токен сразу после конца числового ряда предыдущего виджета.
- **Выравнивание плана**: по умолчанию по последним датам; для Дэшей с "УБ" — по первым.
- **Недельный срез**: есть, если после "нед" идёт строка хотя бы с одной цифрой (не прочерк).

In [ ]:
def parse_widgets(text: str, dash_name: str = ""):
    '''
    Разбирает текст, скопированный с экрана, на отдельные виджеты (показатели).
    Возвращает список словарей:
      {"title", "forecast", "weekly", "periods": [(тип, год, номер), ...], "fact": [...], "plan": [...]}

    dash_name используется только для выбора правила определения ПЕРВОГО заголовка:
      - если в dash_name есть "CB" или "core" - заголовок это строка перед первой строкой с цифрой;
      - иначе (в т.ч. "УБ") - строка перед первым служебным словом (единица/Прогноз/План/Факт/Вып/Дельта).
    '''
    tokens = flatten_tokens(text)
    n = len(tokens)
    widgets = []

    use_digit_rule = ("CB" in dash_name) or ("core" in dash_name)

    pending_meta = []
    current = None
    current_year = None
    awaiting_title = True
    first_widget = True

    def new_widget(title):
        return {"title": title, "forecast": False, "weekly": False,
                "periods": [], "fact": [], "plan": []}

    def check_forecast(widget, token):
        if widget is not None and "прогноз" in token.lower():
            widget["forecast"] = True

    def check_weekly(widget, idx):
        if widget is None:
            return
        tok = tokens[idx]
        if "нед" in tok.lower():
            nxt = tokens[idx + 1] if idx + 1 < n else None
            if nxt is not None and HAS_DIGIT_RE.search(nxt):
                widget["weekly"] = True

    def is_first_title_trigger(token: str) -> bool:
        if use_digit_rule:
            return bool(HAS_DIGIT_RE.search(token))
        return is_label_or_unit_line(token)

    i = 0
    while i < n:
        tok = tokens[i]

        if awaiting_title:
            if first_widget:
                if is_first_title_trigger(tok) and pending_meta:
                    if current and current["periods"]:
                        widgets.append(current)
                    current = new_widget(pending_meta[-1])
                    current_year = None
                    check_forecast(current, tok)
                    check_weekly(current, i)
                    pending_meta = []
                    awaiting_title = False
                    first_widget = False
                    if use_digit_rule:
                        # правило CB/core: триггерный токен (первая строка с цифрой) - это
                        # и есть начало периода, его нельзя "съедать", обрабатываем заново
                        continue
                    i += 1
                    continue
            else:
                if current and current["periods"]:
                    widgets.append(current)
                current = new_widget(tok)
                current_year = None
                check_forecast(current, tok)
                check_weekly(current, i)
                awaiting_title = False
                i += 1
                continue

        parsed, y2 = (None, current_year)
        if current is not None:
            parsed, y2 = parse_period_token(tok, current_year)

        if parsed is not None:
            periods_buf = [parsed]
            current_year = y2
            i += 1
            while i < n:
                p2, y3 = parse_period_token(tokens[i], current_year)
                if p2 is None or p2[0] != parsed[0]:
                    break
                periods_buf.append(p2)
                current_year = y3
                i += 1
            current["periods"] = periods_buf

            def consume_numbers(limit):
                nonlocal i
                vals = []
                while i < n and len(vals) < limit and NUMBER_RE.match(tokens[i]):
                    vals.append(to_number(tokens[i]))
                    i += 1
                return vals

            current["fact"] = consume_numbers(len(periods_buf))
            current["plan"] = consume_numbers(len(periods_buf))
            awaiting_title = True
            continue

        check_forecast(current, tok)
        check_weekly(current, i)
        pending_meta.append(tok)
        i += 1

    if current and current["periods"]:
        widgets.append(current)

    return widgets

### Проверка парсера на реальных примерах и новых правилах (CB/core, УБ)

In [ ]:
_sample_text = '''Обзор
Монитор бизнеса
Цели
ДИб
БИБ
ХУБ
Чистая прибыль
Млн руб
Прогноз
5.0
План
67)
Дельта нед.
-0.1
Факт/прогноз
План
Вып.
Авг2025
Сен
Окт
Ноя
Дек
Янв2026
Фев
Мар
Апр
Май
Июн
Июл
Авг
Сен
Окт
Ноя
Дек
3,4
3,5
3,6
3.7
3.8
3.9
4.2
4.3
4.4
4.5
4.5
4.6
5.7
5.3
4.7
5.3
5.4
98%
97%
95%
44%
85%
86%
88%
89%
CSI
Прогноз
План
Вып-е
114%
нед.
Факт
3Q2025
4Q
1Q2026
2Q
3Q
10.0
11.6
14.6
15.7
16.5
45%
65%
86%
77%
88%
'''

_test_widgets = parse_widgets(_sample_text, dash_name="ХУБ")
for w in _test_widgets:
    print("Показатель:", w["title"])
    print("  Прогноз:", w["forecast"], "| Недельный срез:", w["weekly"])
    print("  Гранулярность:", "Месяц" if w["periods"][0][0] == "M" else "Квартал")
    print("  Периодов:", len(w["periods"]), "| Факт:", len(w["fact"]), "| План:", len(w["plan"]))
    print("  Первый период:", format_period(w["periods"][0]), "Последний:", format_period(w["periods"][-1]))
    print()

# --- проверка правила "CB/core": заголовок = строка перед первой строкой с цифрой ---
_sample_cb = '''Some Menu Item
Another Menu Item
Reactivnost CB
1Q2025
2Q
3Q
1.5
1.7
1.9
'''
_w_cb = parse_widgets(_sample_cb, dash_name="CB_dashboard")
print("CB-правило, показатель:", _w_cb[0]["title"], "(ожидаем 'Reactivnost CB')")

# --- проверка правила "УБ": остаётся старое (по единице измерения / служебным словам) ---
_sample_ub = '''Меню
Показатель с УБ
Млн руб
Факт
Авг2025
Сен
1,0
1,1
'''
_w_ub = parse_widgets(_sample_ub, dash_name="XУБ")
print("УБ-правило, показатель:", _w_ub[0]["title"], "(ожидаем 'Показатель с УБ')")

## Сбор всех файлов и выгрузка в JSON

In [ ]:
TXT_ROOT = Path("txt")
OUTPUT_JSON = Path("combined_widgets.json")


def format_period(p):
    ptype, year, num = p
    if ptype == "M":
        return f"{num:02d}.{year}"
    return f"{num}кв.{year}"


def compute_actual_period(periods, fact, today=None):
    today = today or date.today()
    today_q = (today.month - 1) // 3 + 1
    dated = [p for p, val in zip(periods, fact) if val is not None]
    if not dated:
        return None

    def is_past_or_present(p):
        ptype, year, num = p
        if ptype == "M":
            return (year, num) <= (today.year, today.month)
        return (year, num) <= (today.year, today_q)

    past = [p for p in dated if is_past_or_present(p)]
    pool = past if past else dated
    return max(pool, key=lambda p: (p[1], p[2]))


def align_plan_to_periods(periods, plan, is_ub_dash: bool):
    if not plan:
        return {}
    if is_ub_dash:
        plan_periods = periods[:len(plan)]
    else:
        plan_periods = periods[len(periods) - len(plan):]
    return dict(zip(plan_periods, plan))


if not TXT_ROOT.exists():
    raise FileNotFoundError(f"Папка {TXT_ROOT.resolve()} не найдена")

txt_files = sorted(TXT_ROOT.glob("**/*.txt"))
print(f"Найдено txt-файлов: {len(txt_files)}")

result = []

for fpath in txt_files:
    stem = fpath.stem
    dash, _, screen = stem.partition("_")
    is_ub_dash = "УБ" in dash

    text = fpath.read_text(encoding="utf-8")
    widgets = parse_widgets(text, dash_name=dash)

    if not widgets:
        print(f"⚠️ Ни одного виджета не распознано в {fpath} - проверьте формат файла")

    widgets_out = []
    for w in widgets:
        periods = w["periods"]
        fact = w["fact"]
        plan = w["plan"]

        if not periods or not fact:
            continue

        plan_dict = align_plan_to_periods(periods, plan, is_ub_dash)
        actual_period = compute_actual_period(periods, fact)

        values = []
        for idx, p in enumerate(periods):
            values.append({
                "period": format_period(p),
                "fact": fact[idx] if idx < len(fact) else None,
                "plan": plan_dict.get(p),
            })

        widgets_out.append({
            "title": w["title"],
            "granularity": "Месяц" if periods[0][0] == "M" else "Квартал",
            "forecast": w["forecast"],
            "weekly": w["weekly"],
            "actual_period": format_period(actual_period) if actual_period else None,
            "values": values,
        })

    result.append({
        "dash": dash,
        "screen": screen,
        "source_file": str(fpath),
        "widgets": widgets_out,
    })

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

total_widgets = sum(len(item["widgets"]) for item in result)
print(f"Файлов обработано: {len(result)}")
print(f"Виджетов всего: {total_widgets}")
print(f"Сохранено: {OUTPUT_JSON.resolve()}")